# FuncADL ServiceX Tutorial — Solutions
Worked solutions to the problems in `funcadl_tutorial.ipynb`. Each solution is self-contained: run the setup cell below first, then any problem.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
from servicex import deliver, dataset
from servicex_analysis_utils import to_awk
from func_adl_servicex_xaodr25 import FuncADLQueryPHYSLITE
ds = dataset.Rucio("mc20_13TeV:DAOD_PHYSLITE.38191209._000001.pool.root.1")

## Problem 1: Muons
Same structure as Example 1, swapping `e.Jets()` for `e.Muons()`.

In [ ]:
base_query = FuncADLQueryPHYSLITE()
muons_per_event = (base_query
   .Select(lambda e: e.Muons())
   .Select(lambda muons: {
       'pt': muons.Select(lambda m: m.pt() / 1000),
       'eta': muons.Select(lambda m: m.eta()),
   })
)
spec = {
   'Sample': [{
       'Name': 'Problem1_Muons',
       'Dataset': ds,
       'Query': muons_per_event
   }]
}
muon_data = to_awk(deliver(spec))['Problem1_Muons']
plt.hist(ak.flatten(muon_data.pt), bins=100, range=(0, 200))
plt.xlabel('Muon $p_T$ [GeV]')
plt.ylabel('Number of muons')
plt.title('Problem 1: Muon $p_T$')
plt.show()

## Problem 2: Central Jets
An object-level cut: `.Where()` is applied to the jet container inside the first `.Select()`, so only jets with $|\eta| < 1.0$ are passed on.

In [ ]:
base_query = FuncADLQueryPHYSLITE()
central_jets = (base_query
   .Select(lambda e: e.Jets().Where(lambda j: abs(j.eta()) < 1.0))
   .Select(lambda jets: {
       'pt': jets.Select(lambda j: j.pt() / 1000),
   })
)
spec = {
   'Sample': [{
       'Name': 'Problem2_CentralJets',
       'Dataset': ds,
       'Query': central_jets
   }]
}
central_data = to_awk(deliver(spec))['Problem2_CentralJets']
plt.hist(ak.flatten(central_data.pt), bins=100, range=(0, 200))
plt.xlabel('Jet $p_T$ [GeV]')
plt.ylabel('Number of jets')
plt.title('Problem 2: Central jet ($|\\eta| < 1$) $p_T$')
plt.show()

## Problem 3: Skim and Select
An event-level cut: the `.Where()` before the first `.Select()` drops any event that doesn't have at least 2 jets above 40 GeV. `.Count()` counts the jets that pass the inner cut.

In [ ]:
base_query = FuncADLQueryPHYSLITE()
dijet_events = (base_query
   .Where(lambda e: e.Jets().Where(lambda j: j.pt() / 1000 > 40).Count() >= 2)
   .Select(lambda e: e.Jets())
   .Select(lambda jets: {
       'pt': jets.Select(lambda j: j.pt() / 1000),
       'eta': jets.Select(lambda j: j.eta()),
   })
)
spec = {
   'Sample': [{
       'Name': 'Problem3_Dijets',
       'Dataset': ds,
       'Query': dijet_events
   }]
}
dijet_data = to_awk(deliver(spec))['Problem3_Dijets']
plt.hist(ak.flatten(dijet_data.pt), bins=100, range=(0, 200))
plt.xlabel('Jet $p_T$ [GeV]')
plt.ylabel('Number of jets')
plt.title('Problem 3: Jet $p_T$ in dijet events')
plt.show()
print(f'Events surviving the skim: {len(dijet_data.pt)}')